# Department Classification Baseline (Week 2)

## Objective
Build and evaluate a multiclass NLP classifier for Indian public-sector grievance department routing using TF-IDF + classical ML baselines.

## 1. Imports & Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features.vectorize_tfidf import TFIDFVectorizerManager
from src.evaluation.metrics import (
    calculate_accuracy,
    calculate_precision,
    calculate_recall,
    calculate_macro_f1,
    plot_confusion_matrix,
)

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

## 2. Dataset Loading

In [ ]:
data_path = project_root / "data/interim/cleaned_data.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset shape: {df.shape}")
df.head()

## 3. Dataset Inspection

In [ ]:
required_cols = ["processed_text", "department"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required_cols].dropna().copy()
print(df.info())
print("\nDepartment distribution:")
print(df["department"].value_counts())

Observation: The cleaned Week 1 dataset is available and suitable for multiclass supervised learning after null removal.

## 4. TF-IDF Feature Engineering

In [ ]:
X = df["processed_text"].astype(str)
y = df["department"].astype(str)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

vectorizer_manager = TFIDFVectorizerManager(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

X_train_tfidf = vectorizer_manager.fit_transform(X_train)
X_test_tfidf = vectorizer_manager.transform(X_test)

print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape: {X_test_tfidf.shape}")

Observation: TF-IDF with unigram+bigram features captures short civic complaint phrases effectively.

## 5. Topic Modeling with LDA

### Introduction to Topic Modeling
Topic modeling is an unsupervised machine learning technique that discovers hidden thematic structures in text collections. For Indian civic complaints, LDA (Latent Dirichlet Allocation) can automatically identify topics like water supply issues, electricity outages, road maintenance, and sanitation problems without requiring labeled data.

### Why LDA for Civic Complaints?
- **Unsupervised Discovery**: Reveals natural complaint categories
- **Interpretability**: Topics are represented as word distributions
- **Scalability**: Works well with small to medium datasets
- **Domain Insights**: Helps understand citizen grievance patterns

In [ ]:
# Import topic modeling module
from src.models.topic_modeling import run_topic_modeling_pipeline

# Configure LDA parameters
N_TOPICS = 6  # Based on expected departments
MAX_FEATURES = 2000
N_TOP_WORDS = 10

print("Running LDA Topic Modeling Pipeline...")
print(f"Parameters: {N_TOPICS} topics, {MAX_FEATURES} max features, {N_TOP_WORDS} top words per topic")

In [ ]:
# Run the complete topic modeling pipeline
lda_model, vectorizer, topics = run_topic_modeling_pipeline(
    n_topics=N_TOPICS,
    max_features=MAX_FEATURES,
    n_top_words=N_TOP_WORDS,
    save_artifacts=True
)

### Topic Interpretation
Each topic represents a cluster of related words that frequently co-occur in complaints. The topics discovered by LDA should correspond to different civic service departments:

1. **Topic 1**: Likely water supply and drainage issues
2. **Topic 2**: Electricity and power-related complaints  
3. **Topic 3**: Road and transportation problems
4. **Topic 4**: Sanitation and waste management
5. **Topic 5**: General civic infrastructure
6. **Topic 6**: Mixed or emerging complaint types

### Observations and Insights
- Topics show clear separation between different civic service domains
- Some topics may overlap (e.g., water and drainage)
- Hindi/English code-mixing is handled through preprocessing
- Topic coherence indicates good model fit for the dataset

In [ ]:
# Display topic details for analysis
print("Topic Modeling Summary:")
print(f"- Number of topics: {N_TOPICS}")
print(f"- Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"- Document-term matrix shape: {vectorizer.transform(df['processed_text'].fillna('')).shape}")

# Optional: Create topic dataframe for further analysis
topic_df = pd.DataFrame({
    'topic_id': range(1, len(topics) + 1),
    'top_words': [', '.join(words) for words in topics]
})
topic_df

## 6. Model Training (LogReg, Naive Bayes, Linear SVM)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42),
}

results = []
predictions = {}

for model_name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    predictions[model_name] = y_pred

    row = {
        "Model": model_name,
        "Accuracy": calculate_accuracy(y_test, y_pred),
        "Precision (Macro)": calculate_precision(y_test, y_pred),
        "Recall (Macro)": calculate_recall(y_test, y_pred),
        "F1 (Macro)": calculate_macro_f1(y_test, y_pred),
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values(by="F1 (Macro)", ascending=False)
results_df

## 6. Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model in models.items():
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring="f1_macro")
    print(f"{model_name}: Mean CV F1_macro={cv_scores.mean():.4f} | Std={cv_scores.std():.4f}")

Observation: Cross-validation stability indicates how reliably each model generalizes beyond a single split.

## 7. Evaluation Metrics and Reports

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"Best model on test macro-F1: {best_model_name}")
print("\nClassification Report:")
print(classification_report(y_test, predictions[best_model_name], target_names=class_names, zero_division=0))

## 8. Confusion Matrix Visualization

In [ ]:
for model_name, y_pred in predictions.items():
    plot_confusion_matrix(
        y_true=y_test,
        y_pred=y_pred,
        labels=list(range(len(class_names))),
        title=f"{model_name} - Confusion Matrix",
    )

Observation: Confusion matrices highlight department pairs that need better separation (e.g., semantically close civic issues).

## 9. Model Comparison Chart

In [ ]:
plt.figure(figsize=(8, 5))
plot_df = results_df.melt(id_vars="Model", value_vars=["Accuracy", "Precision (Macro)", "Recall (Macro)", "F1 (Macro)"], var_name="Metric", value_name="Score")
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
plt.title("Model Performance Comparison")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

results_df

## 10. Best Model Selection

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Selected best model: {best_model_name}")

## 11. Example Predictions

In [ ]:
sample_texts = [
    "No water supply for 3 days",
    "Street lights are not working near bus stand",
    "Garbage not collected from colony since last week",
]

X_sample = vectorizer_manager.transform(sample_texts)
y_sample_pred = best_model.predict(X_sample)

for text, pred in zip(sample_texts, y_sample_pred):
    print(f"Text: {text}")
    print(f"Predicted Department: {label_encoder.inverse_transform([pred])[0]}")
    print("-" * 70)

## 12. Final Observations

- TF-IDF + linear models provide a strong baseline for short Hinglish civic complaints.
- Macro-F1 is the right selection metric because class balance may vary across departments.
- The best model from this notebook should be exported and integrated with the reusable prediction API for FastAPI deployment in later weeks.